# FT105A — Trabalho 1: Multidimensional Visualization of PNAD Contínua (2026Q2)

**Authors:** Étore Braga e Santos, Raphael Pizzi, Saulo Celson Bergantini Dias

**Course:** FT105A — Information Visualization (PPGT)  
**Delivery format:** static GRIVAPP 2027 PDF (figures designed for a fixed frame)  
**Data:** PNAD Contínua microdata, 2nd quarter 2026 (`PNADC_022026`)

This notebook is the **academic content outline** for the camera-ready report. It reuses the group's weighted analysis and exploratory notebooks; it does not invent IBGE estimates.

### Official graded trio (locked)

1. **Bubble scatter** — UF informality × income  
2. **Heatmap small-multiples** — unemployment by age × sex × colour/race  
3. **Sankey** — labour-force status ages 25–49 (default/static frame: all schooling levels)

Exploratory techniques (parallel coordinates, treemap, pixel matrix, SPLOM, parallel sets, RadViz) appear only in the **Appendix**.


## 1. Introduction

Information Visualization (InfoVis) supports discovery of structure in multivariate social statistics. For Trabalho 1 we analyse PNAD Contínua microdata for 2026Q2 with three distinct visual techniques, each mapping **more than three** data variables to graphical channels (Ward / Grinstein / Keim families as in course Topic 04, and visual-mapping guidelines from Topic 03).

The graded narrative is **static**: each main figure must remain readable when exported as PNG for the GRIVAPP PDF. Interactive Plotly menus (e.g. schooling filter on the Sankey) are optional exploration aids; the default frame is the one intended for print.


## 2. The PNAD Contínua data

IBGE's Continuous National Household Sample Survey (PNAD Contínua) provides quarterly labour-market indicators for Brazil. We use the **2nd quarter of 2026** microdata and the official documentation dictionary/input.

**Preprocessing pipeline** (scripts under `code/`):

1. `01_download.py` — fetch zip + dictionary  
2. `02_import_pnadc.py` — fixed-width import (selected columns)  
3. `03_prepare_sample.py` — ~50k-row sample for exploration  
4. `04_explore_viz.py` — optional exploratory HTML  
5. `05_prepare_extrato.py` — **63-column** weighted extract for the official trio  

All rates, means and medians in the graded figures are weighted by the person weight **`V1028`**. Published IBGE release benchmarks for 2026Q2 used as sanity checks: unemployment **5.4%**, informality **37.4%**, mean usual income **R$ 3,738**.

> Run `python 05_prepare_extrato.py` before the cells below if `pnadc_2026q2_extrato.parquet` is missing (processed data are gitignored).


## 3. Setup and data load


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

DATA = Path("../../../data/processed").resolve()
FIGURES = Path("../../../entregas/figures").resolve()
FIGURES.mkdir(parents=True, exist_ok=True)
APPENDIX_DIR = Path("output/figures").resolve()
APPENDIX_DIR.mkdir(parents=True, exist_ok=True)

EXTRATO = DATA / "pnadc_2026q2_extrato.parquet"
SAMPLE = DATA / "pnadc_2026q2_sample.parquet"

def save_static(fig: go.Figure, path: Path, width: int = 1100, height: int = 700) -> None:
    """Export PNG for the GRIVAPP PDF. Requires kaleido; otherwise saves HTML fallback."""
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(path), width=width, height=height, scale=2)
        print(f"saved {path}")
    except Exception as exc:  # noqa: BLE001
        html_fallback = path.with_suffix(".html")
        fig.write_html(str(html_fallback), include_plotlyjs="cdn")
        warnings.warn(
            f"PNG export failed ({exc!r}). Saved HTML fallback at {html_fallback}. "
            "Install kaleido (`pip install kaleido`) and re-run for PNG."
        )

print("DATA =", DATA)
print("extrato exists:", EXTRATO.exists())
print("sample exists:", SAMPLE.exists())


DATA = /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/data/processed
extrato exists: True
sample exists: True


In [2]:
if not EXTRATO.exists():
    raise FileNotFoundError(
        f"Missing {EXTRATO}.\n"
        "Generate it first:\n"
        "  python 05_prepare_extrato.py\n"
        "Do not invent IBGE numbers; re-run this notebook after the extract is ready."
    )

pn = pd.read_parquet(EXTRATO)

UF_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}
REGIAO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}
ORDEM_REG = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
COR_REGIAO = {
    "Norte": "#009E73", "Nordeste": "#D55E00", "Sudeste": "#0072B2",
    "Sul": "#CC79A7", "Centro-Oeste": "#E69F00",
}

pn["uf"] = pn["UF"].map(UF_SIGLA)
pn["regiao"] = pn["UF"].str[0].map(REGIAO)
pn["sexo"] = pn["V2007"].map({1: "Homem", 2: "Mulher"})
pn["cor"] = pn["V2010"].map({1: "Branca", 2: "Negra", 4: "Negra"})
pn["peso"] = pn["V1028"]
pn["renda"] = pn["VD4019"]
pn["horas"] = pn["VD4031"].astype(float)
pn["ocupado"] = pn["VD4002"].eq(1).fillna(False).astype(bool)
pn["na_forca"] = pn["VD4001"].eq(1).fillna(False).astype(bool)

pos = pn["VD4009"]
sem_cnpj = pn["V4019"].eq(2).fillna(False)
# IBGE informality definition (public employees without carteira excluded)
pn["informal"] = (
    pn["ocupado"] & (pos.isin([2, 4, 10]) | (pos.isin([8, 9]) & sem_cnpj))
).astype(bool)

pn["dom_id"] = pn["UPA"] + pn["V1008"] + pn["V1014"]
criancas_por_dom = pn.loc[pn["V2009"] <= 5].groupby("dom_id").size()
pn["crianca_0a5"] = (pn["dom_id"].map(criancas_por_dom).fillna(0) > 0).astype(bool)


def pct(mask, w):
    return 100 * w[mask].sum() / w.sum()


def media_pond(v, w):
    ok = v.notna()
    return float(np.average(v[ok], weights=w[ok])) if ok.any() else np.nan


def mediana_pond(v, w):
    ok = v.notna()
    v, w = v[ok].to_numpy(dtype=float), w[ok].to_numpy(dtype=float)
    ordem = np.argsort(v)
    acum = np.cumsum(w[ordem])
    return float(v[ordem][np.searchsorted(acum, acum[-1] / 2)])


def gini_pond(v, w):
    ok = v.notna()
    v, w = v[ok].to_numpy(dtype=float), w[ok].to_numpy(dtype=float)
    ordem = np.argsort(v)
    v, w = v[ordem], w[ordem]
    acum_w = np.cumsum(w)
    acum_vw = np.cumsum(v * w)
    return float(1 - 2 * np.sum((acum_vw - v * w / 2) * w) / (acum_w[-1] * acum_vw[-1]))


forca = pn[pn["na_forca"]]
ocup = pn[pn["ocupado"]]
print(f"{len(pn):,} persons | estimated population {pn['peso'].sum() / 1e6:.1f} million")
print(
    f"unemployment {pct(forca['VD4002'].eq(2), forca['peso']):.1f}% | "
    f"informality {pct(ocup['informal'], ocup['peso']):.1f}% | "
    f"mean income R$ {media_pond(ocup['renda'], ocup['peso']):,.0f}"
)


521,730 persons | estimated population 213.5 million
unemployment 5.4% | informality 37.4% | mean income R$ 3,738


## 4. Technique I — Bubble scatter: geography of informality

**Family:** point mark with size and colour channels (multivariate scatter).

**Visual mapping**

| Channel | Variable |
| --- | --- |
| x | Informality rate (% of employed; IBGE definition) |
| y | Mean usual labour income (R$/month) — static default |
| marker area | Number of employed persons (weighted) |
| colour | Macro-region |
| text | UF abbreviation |

Additional attributes (Gini, unemployment, tertiary education) remain available on hover in the interactive notebook but are **not required** to read the static PNG.

**Interpretation (for the PDF).** Informality and income move strongly in opposite directions across the 27 UFs. States such as Santa Catarina combine low informality with high income; Maranhão sits at the opposite corner. Several North/Northeast UFs share a median income equal to the national minimum wage — a floor visible when the y-axis is switched to the median in exploratory mode. The Federal District is an outlier with high income, low informality, and the highest Gini, which the bubble alone does not encode.


In [3]:
linhas = []
for uf, g in pn.groupby("uf"):
    o = g[g["ocupado"]]
    f = g[g["na_forca"]]
    linhas.append({
        "uf": uf,
        "regiao": g["regiao"].iloc[0],
        "ocupados_mil": o["peso"].sum() / 1e3,
        "informalidade": pct(o["informal"], o["peso"]),
        "renda_mediana": mediana_pond(o["renda"], o["peso"]),
        "renda_media": media_pond(o["renda"], o["peso"]),
        "renda_hora": media_pond(o["renda"] / (o["horas"] * 4.33).where(o["horas"] > 0), o["peso"]),
        "gini": gini_pond(o["renda"], o["peso"]),
        "desocupacao": pct(f["VD4002"].eq(2), f["peso"]),
        "superior": pct(o["VD3004"].eq(7), o["peso"]),
    })
ufs = pd.DataFrame(linhas)

r_mean = ufs["informalidade"].corr(ufs["renda_media"])
print("corr(informality, mean income) =", round(r_mean, 2))


def fig_bolhas_uf(ufs: pd.DataFrame) -> go.Figure:
    """Static-friendly bubble scatter (mean income on y; no updatemenus)."""
    fig = go.Figure()
    for reg in ORDEM_REG:
        d = ufs[ufs["regiao"] == reg]
        fig.add_trace(go.Scatter(
            x=d["informalidade"], y=d["renda_media"], name=reg,
            mode="markers+text", text=d["uf"], textposition="top center",
            textfont=dict(size=11),
            marker=dict(
                size=d["ocupados_mil"], sizemode="area", sizemin=5,
                sizeref=2 * ufs["ocupados_mil"].max() / 60 ** 2,
                color=COR_REGIAO[reg], opacity=0.85, line=dict(width=1, color="white"),
            ),
            hovertemplate=(
                "<b>%{text}</b> (" + reg + ")"
                "<br>informality: %{x:.1f}%"
                "<br>mean income: R$ %{y:,.0f}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=f"Informality × mean labour income by UF, 2026Q2 (area = employed; r = {r_mean:.2f})",
        xaxis_title="Informality (% of employed)",
        yaxis_title="Mean income (R$/month)",
        legend_title_text="Region",
        height=620, margin=dict(t=80),
        template="plotly_white",
    )
    fig.update_xaxes(ticksuffix="%")
    return fig


fig_bubbles = fig_bolhas_uf(ufs)
fig_bubbles.show()
save_static(fig_bubbles, FIGURES / "01_bubble_informality_income.png", width=1100, height=650)


corr(informality, mean income) = -0.85


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/entregas/figures/01_bubble_informality_income.png


## 5. Technique II — Heatmap small-multiples: unemployment by age, sex and colour/race

**Family:** region / matrix display with small multiples.

**Visual mapping**

| Channel | Variable |
| --- | --- |
| row | Age band |
| column | Sex × colour/race (Black = preto + pardo, IBGE practice) |
| cell colour | Unemployment rate (%) |
| panel | Brasil + five macro-regions |

Cells with fewer than 30 sample records are left blank (rate not estimable).

**Interpretation (for the PDF).** Black women show higher unemployment than white men in almost all age×region cells with sufficient sample size; age dominates the level (adolescents high, ages 50+ low). The largest black-women / white-men contrast appears at ages 25–29. Ordering of the two middle groups (black men, white women) is not stable across panels — only the extremes remain consistent.


In [4]:
FAIXAS = [14, 18, 25, 30, 40, 50, 60, 200]
ROTULO_FAIXA = ["14-17", "18-24", "25-29", "30-39", "40-49", "50-59", "60+"]
GRUPOS = ["Homem branco", "Homem negro", "Mulher branca", "Mulher negra"]
NOME_GRUPO = {
    ("Homem", "Branca"): "Homem branco", ("Homem", "Negra"): "Homem negro",
    ("Mulher", "Branca"): "Mulher branca", ("Mulher", "Negra"): "Mulher negra",
}

forca_hm = pn[pn["na_forca"] & pn["cor"].notna()].copy()
forca_hm["faixa"] = pd.cut(forca_hm["V2009"].astype(int), FAIXAS, right=False, labels=ROTULO_FAIXA)
forca_hm["grupo"] = [NOME_GRUPO[(s, c)] for s, c in zip(forca_hm["sexo"], forca_hm["cor"])]
forca_hm["peso_desocupado"] = forca_hm["peso"] * forca_hm["VD4002"].eq(2).astype(float)


def taxa_por_celula(d: pd.DataFrame) -> pd.DataFrame:
    t = (
        d.groupby(["faixa", "grupo"], observed=True)
        .agg(peso=("peso", "sum"), peso_desocupado=("peso_desocupado", "sum"), n=("peso", "size"))
        .reset_index()
    )
    t["taxa"] = 100 * t["peso_desocupado"] / t["peso"]
    t.loc[t["n"] < 30, "taxa"] = np.nan
    return t


paineis = {"Brasil": taxa_por_celula(forca_hm)}
for reg in ORDEM_REG:
    paineis[reg] = taxa_por_celula(forca_hm[forca_hm["regiao"] == reg])


def fig_calor_desocupacao(paineis: dict) -> go.Figure:
    fig = make_subplots(
        rows=2, cols=3, subplot_titles=list(paineis), shared_yaxes=True,
        horizontal_spacing=0.04, vertical_spacing=0.16,
    )
    for i, (nome, t) in enumerate(paineis.items()):
        z = t.pivot(index="faixa", columns="grupo", values="taxa").reindex(index=ROTULO_FAIXA, columns=GRUPOS).astype(float)
        n = t.pivot(index="faixa", columns="grupo", values="n").reindex(index=ROTULO_FAIXA, columns=GRUPOS)
        forca_mil = t.pivot(index="faixa", columns="grupo", values="peso").reindex(index=ROTULO_FAIXA, columns=GRUPOS) / 1e3
        texto = np.where(np.isnan(z.values), "", np.round(z.values, 1).astype(str))
        fig.add_trace(go.Heatmap(
            z=z.values, x=GRUPOS, y=ROTULO_FAIXA, coloraxis="coloraxis", xgap=2, ygap=2,
            text=texto, texttemplate="%{text}", textfont=dict(size=11),
            customdata=np.dstack([n.values, forca_mil.values]),
            hovertemplate=(
                nome + "<br>%{y}, %{x}<br>unemployment: %{z:.1f}%"
                "<br>labour force: %{customdata[1]:,.0f}k<br>n: %{customdata[0]}<extra></extra>"
            ),
        ), row=i // 3 + 1, col=i % 3 + 1)

    fig.update_layout(
        title="Unemployment rate (%) by age, sex and colour/race — Brazil and regions, 2026Q2",
        coloraxis=dict(
            colorscale=[[0, "#eef4fc"], [1, "#2a78d6"]], cmin=0, cmax=25,
            colorbar=dict(title="Unemployment (%)", ticksuffix="%"),
        ),
        height=760, margin=dict(t=90), template="plotly_white",
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(tickangle=-25)
    return fig


fig_heat = fig_calor_desocupacao(paineis)
fig_heat.show()
save_static(fig_heat, FIGURES / "02_heatmap_unemployment.png", width=1200, height=800)


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/entregas/figures/02_heatmap_unemployment.png


## 6. Technique III — Sankey: labour-force participation ages 25–49

**Family:** flow / staged categorical diagram.

**Visual mapping (static default frame)**

| Stage | Variable |
| --- | --- |
| 1 | Sex |
| 2 | Young child (0–5) present in the household |
| 3 | Reference-week status: employed / unemployed / out of the labour force |
| link width | Weighted population (thousands), ages 25–49 |
| colour | Sex × child presence |

**Static PDF recommendation:** export the default frame *all schooling levels*. Schooling filters may be used in the companion weighted notebook for exploration but must not be required to understand the printed figure.

**Interpretation (for the PDF).** Presence of a young child is associated with lower female participation and slightly higher male participation. The magnitude of the female gap varies by schooling and is largest in the intermediate schooling group (fundamental complete to secondary). Reading the width of the “out of the labour force” band per path summarises the finding without interaction.


In [5]:
adultos = pn[(pn["V2009"] >= 25) & (pn["V2009"] <= 49) & pn["sexo"].notna()].copy()
adultos["condicao"] = np.select(
    [adultos["ocupado"].to_numpy(), adultos["VD4002"].eq(2).fillna(False).to_numpy(dtype=bool)],
    ["Employed", "Unemployed"], default="Out of labour force",
)
adultos["escol"] = pd.cut(
    adultos["VD3004"].astype(float), [0, 2, 5, 7],
    labels=["Up to incomplete fundamental", "Fundamental complete to secondary", "Tertiary (incl. incomplete)"],
)

CONDICOES = ["Employed", "Unemployed", "Out of labour force"]
PLURAL = {"Homem": "Men", "Mulher": "Women"}
COMBOS = [("Homem", True), ("Homem", False), ("Mulher", True), ("Mulher", False)]
COR_COMBO = {
    ("Homem", True): "0,114,178", ("Homem", False): "86,180,233",
    ("Mulher", True): "213,94,0", ("Mulher", False): "230,159,0",
}


def fluxos_sankey(d: pd.DataFrame):
    valores, rotulos = [], []
    for sexo in ["Homem", "Mulher"]:
        g = d[d["sexo"] == sexo]
        rotulos.append(f"{PLURAL[sexo]}<br>participation {pct(g['na_forca'], g['peso']):.0f}%")
    for sexo, com in COMBOS:
        g = d[(d["sexo"] == sexo) & (d["crianca_0a5"] == com)]
        valores.append(g["peso"].sum() / 1e3)
        label = "with" if com else "without"
        rotulos.append(f"{label} child 0–5 at home<br>participation {pct(g['na_forca'], g['peso']):.0f}%")
    for sexo, com in COMBOS:
        g = d[(d["sexo"] == sexo) & (d["crianca_0a5"] == com)]
        for cond in CONDICOES:
            valores.append(g.loc[g["condicao"] == cond, "peso"].sum() / 1e3)
    rotulos += CONDICOES
    return valores, rotulos


origem = [0, 0, 1, 1] + [2 + i for i in range(4) for _ in CONDICOES]
destino = [2, 3, 4, 5] + [6 + j for _ in range(4) for j in range(len(CONDICOES))]
cor_link = (
    [f"rgba({COR_COMBO[c]},0.55)" for c in COMBOS]
    + [f"rgba({COR_COMBO[c]},0.45)" for c in COMBOS for _ in CONDICOES]
)
cor_no = ["#0072B2", "#D55E00"] + [f"rgb({COR_COMBO[c]})" for c in COMBOS] + ["#c2c2c2", "#8f8f8f", "#5c5c5c"]

# Static default frame: all schooling levels only (no updatemenus)
valores, rotulos = fluxos_sankey(adultos)
fig_sankey = go.Figure(go.Sankey(
    arrangement="snap", valueformat=",.0f", valuesuffix="k",
    node=dict(label=rotulos, color=cor_no, pad=22, thickness=18, line=dict(width=0)),
    link=dict(source=origem, target=destino, value=valores, color=cor_link),
))
fig_sankey.update_layout(
    title="Ages 25–49: sex → young child at home → labour-force status (all schooling levels), 2026Q2",
    height=620, margin=dict(t=80, l=20, r=20), font=dict(size=12),
    template="plotly_white",
)

participacao = (
    adultos.assign(na_forca_p=adultos["peso"] * adultos["na_forca"])
    .groupby(["escol", "sexo", "crianca_0a5"], observed=True)
    .agg(peso=("peso", "sum"), na_forca_p=("na_forca_p", "sum"))
)
participacao = (100 * participacao["na_forca_p"] / participacao["peso"]).unstack(["sexo", "crianca_0a5"]).round(1)
participacao.columns = [f"{s}, {'with' if c else 'without'} child" for s, c in participacao.columns]
display(participacao)

fig_sankey.show()
save_static(fig_sankey, FIGURES / "03_sankey_labour_force_25_49.png", width=1100, height=650)


,"Homem, without child","Homem, with child","Mulher, without child","Mulher, with child"
escol,,,,
Up to incomplete fundamental,78.3,87.8,48.9,39.1
Fundamental complete to secondary,91.4,95.0,72.4,55.5
Tertiary (incl. incomplete),95.0,98.5,89.2,80.2


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/entregas/figures/03_sankey_labour_force_25_49.png


## 7. Findings (three graded insights)

These statements summarise what the official trio shows; figures above are the evidence. Numbers come from the weighted extract (not invented).

1. **Bubble scatter.** Informality and mean labour income are strongly negatively associated across UFs (correlation on the order of −0.85 with the mean; see printed *r* on the figure title). Geographic extremes (e.g. SC vs MA) and the DF Gini outlier illustrate why area and colour channels matter alongside position.

2. **Heatmap small-multiples.** Disadvantage for black women appears in most age×region cells with adequate sample size; age dominates absolute levels; the largest black-women / white-men ratio is at ages 25–29; after age 60 rates converge. Middle-group ordering is unstable across panels.

3. **Sankey (static default).** A young child at home coincides with lower female and higher male participation among ages 25–49; the female gap is schooling-dependent and largest in the intermediate schooling band. The default “all schooling levels” frame is the one to print.

Detailed Portuguese write-ups and caveats (IBGE informality definition; *n* < 30 blank cells) live in `pnad_analise_ponderada.ipynb` (author: Raphael Pizzi).


## 8. Conclusion

Three complementary techniques — multivariate bubble scatter, small-multiple heatmaps, and a staged Sankey — expose spatial, demographic, and household-structure patterns in Brazil's 2026Q2 labour market without reducing the survey to a single chart type. Designing for a **static** GRIVAPP frame keeps the narrative reproducible in print while preserving Plotly for authoring.

**Next steps for delivery:** place exported PNGs from `entregas/figures/` into the GRIVAPP Word/LaTeX template; keep the Appendix plots out of the main three-insight sections.


---

# Appendix — Exploratory techniques (not the graded trio)

The following figures reuse logic from `pnad_dataviz.ipynb` on the **50k sample** (`pnadc_2026q2_sample.parquet`). They document techniques considered during design (including the draft parallel coordinates / treemap / pixel matrix set) and must be clearly labelled as **appendix / exploration only**.


In [6]:
if not SAMPLE.exists():
    raise FileNotFoundError(
        f"Missing {SAMPLE}. Run python 03_prepare_sample.py before the appendix cells."
    )

df = pd.read_parquet(SAMPLE)
print(df.shape)
df.head(3)


(50000, 14)


,ano,trimestre,uf_cod,uf,sexo,idade,cor_raca,ocupacao_cbo,setor,rend_principal,instrucao,cond_ocupacao,rend_todos,peso
0,2026,2,41,PR,Homem,41,Branca,5222,Comercio e reparacao,2600.0,Fund. completo ou equiv.,Ocupado,2600.0,580.990103
1,2026,2,52,GO,Homem,50,Preta,9212,Agricultura,1200.0,Fund. incompleto ou equiv.,Ocupado,1200.0,96.464521
2,2026,2,35,SP,Mulher,41,Preta,4223,Admin publica,2000.0,Superior completo,Ocupado,2000.0,1362.102335


### A.1–A.6 Exploratory figures

Labels are prefixed **APPENDIX**. These are **not** part of the official graded trio.


In [7]:
import math

INSTR_ORDER = [
    "Sem instrucao",
    "Fund. incompleto ou equiv.",
    "Fund. completo ou equiv.",
    "Medio incompleto ou equiv.",
    "Medio completo ou equiv.",
    "Superior incompleto ou equiv.",
    "Superior completo ou equiv.",
]


def fig_parallel_coords(df: pd.DataFrame) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0) & df["idade"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    if len(d) > 8000:
        d = d.sample(n=8000, random_state=0)
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True).codes
    d = d.loc[d["instr_ord"] >= 0]
    dims = [
        dict(label="Age", values=d["idade"]),
        dict(label="Education (ord.)", values=d["instr_ord"], tickvals=list(range(len(INSTR_ORDER))), ticktext=INSTR_ORDER),
        dict(label="Income (R$)", values=d["rend_todos"]),
    ]
    fig = go.Figure(go.Parcoords(
        line=dict(color=d["idade"], colorscale="Viridis", showscale=True, colorbar=dict(title="Age")),
        dimensions=dims,
    ))
    fig.update_layout(
        title="APPENDIX — Parallel coordinates (employed sample; exploratory)",
        height=520, margin=dict(t=80), template="plotly_white",
    )
    return fig


def fig_treemap(df: pd.DataFrame) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado") & df["setor"].notna()
        & (df["setor"] != "Nao classificado / NA")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
    ].copy()
    g = d.groupby(["uf", "setor"], as_index=False).agg(n=("rend_todos", "size"), rend_medio=("rend_todos", "mean"))
    fig = px.treemap(
        g, path=["uf", "setor"], values="n", color="rend_medio",
        color_continuous_scale="YlGnBu",
        title="APPENDIX — Treemap UF → sector (count; colour = mean income; exploratory)",
    )
    fig.update_layout(height=560, margin=dict(t=80), template="plotly_white")
    return fig


def _to_grid(values: np.ndarray, n_cols: int) -> np.ndarray:
    n = len(values)
    n_rows = int(math.ceil(n / n_cols))
    grid = np.full((n_rows, n_cols), np.nan, dtype=float)
    grid.flat[:n] = values
    return grid


def fig_pixel_matrix(df: pd.DataFrame, n_max: int = 10_000, n_cols: int = 100) -> go.Figure:
    d = df.loc[df["rend_todos"].notna() & (df["rend_todos"] > 0) & df["idade"].notna()].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    if len(d) > n_max:
        d = d.sample(n=n_max, random_state=0)
    d = d.sort_values(["sexo", "idade"])
    z = _to_grid(np.log1p(d["rend_todos"].to_numpy()), n_cols)
    fig = go.Figure(go.Heatmap(z=z, colorscale="Magma", colorbar=dict(title="log(1+income)")))
    fig.update_layout(
        title="APPENDIX — Pixel-oriented matrix (log income; sorted by sex/age; exploratory)",
        height=420, margin=dict(t=80), template="plotly_white",
        xaxis_title="column", yaxis_title="row",
    )
    return fig


def fig_splom(df: pd.DataFrame, n_max: int = 4000) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
        & df["idade"].notna() & df["instrucao"].notna() & df["sexo"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True).codes
    d = d.loc[d["instr_ord"] >= 0]
    if len(d) > n_max:
        d = d.sample(n=n_max, random_state=0)
    fig = px.scatter_matrix(
        d, dimensions=["idade", "instr_ord", "rend_todos"], color="sexo",
        labels={"idade": "Age", "instr_ord": "Educ.", "rend_todos": "Income"},
        title="APPENDIX — SPLOM (exploratory)",
    )
    fig.update_traces(diagonal_visible=False, marker=dict(size=3, opacity=0.35))
    fig.update_layout(height=620, margin=dict(t=80), template="plotly_white")
    return fig


def fig_parallel_sets(df: pd.DataFrame) -> go.Figure:
    d = df.loc[df["sexo"].notna() & df["cor_raca"].notna() & df["instrucao"].notna() & df["cond_ocupacao"].notna()].copy()
    mapa = {
        "Sem instrucao": "Ate fund.",
        "Fund. incompleto ou equiv.": "Ate fund.",
        "Fund. completo ou equiv.": "Ate fund.",
        "Medio incompleto ou equiv.": "Medio",
        "Medio completo ou equiv.": "Medio",
        "Superior incompleto ou equiv.": "Superior",
        "Superior completo ou equiv.": "Superior",
    }
    d["escol4"] = d["instrucao"].map(mapa)
    d = d.dropna(subset=["escol4"])
    stages = ["sexo", "cor_raca", "escol4", "cond_ocupacao"]
    labels = []
    for s in stages:
        labels.extend(sorted(d[s].unique().tolist()))
    label_index = {lab: i for i, lab in enumerate(labels)}
    sources, targets, values = [], [], []
    for a, b in zip(stages[:-1], stages[1:]):
        g = d.groupby([a, b]).size().reset_index(name="n")
        for _, row in g.iterrows():
            sources.append(label_index[row[a]])
            targets.append(label_index[row[b]])
            values.append(int(row["n"]))
    fig = go.Figure(go.Sankey(
        node=dict(label=labels, pad=12, thickness=14),
        link=dict(source=sources, target=targets, value=values),
    ))
    fig.update_layout(
        title="APPENDIX — Parallel sets / categorical flow (exploratory)",
        height=520, margin=dict(t=80), template="plotly_white",
    )
    return fig


def _radviz_xy(D: np.ndarray):
    angles = np.linspace(0, 2 * np.pi, D.shape[1], endpoint=False)
    A = np.column_stack([np.cos(angles), np.sin(angles)])
    num = D @ A
    den = D.sum(axis=1, keepdims=True)
    den = np.where(den == 0, np.nan, den)
    P = num / den
    return P[:, 0], P[:, 1]


def fig_radviz(df: pd.DataFrame, n_max: int = 5000) -> go.Figure:
    d = df.loc[
        (df["cond_ocupacao"] == "Ocupado")
        & df["rend_todos"].notna() & (df["rend_todos"] > 0)
        & df["idade"].notna() & df["instrucao"].notna() & df["sexo"].notna()
    ].copy()
    d = d.loc[d["rend_todos"] <= d["rend_todos"].quantile(0.99)]
    d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True).codes.astype(float)
    d = d.loc[d["instr_ord"] >= 0]
    if len(d) > n_max:
        d = d.sample(n=n_max, random_state=0)
    cols = ["idade", "instr_ord", "rend_todos"]
    M = d[cols].to_numpy(dtype=float)
    mn, mx = M.min(axis=0), M.max(axis=0)
    D = (M - mn) / np.where(mx - mn == 0, 1, mx - mn)
    x, y = _radviz_xy(D)
    fig = go.Figure()
    for sexo, g in pd.DataFrame({"x": x, "y": y, "sexo": d["sexo"].to_numpy()}).groupby("sexo"):
        fig.add_trace(go.Scatter(
            x=g["x"], y=g["y"], mode="markers", name=sexo, marker=dict(size=4, opacity=0.35),
        ))
    angles = np.linspace(0, 2 * np.pi, len(cols), endpoint=False)
    ax, ay = np.cos(angles), np.sin(angles)
    fig.add_trace(go.Scatter(
        x=ax, y=ay, mode="markers+text", text=cols, textposition="top center",
        marker=dict(size=10, color="black"), name="anchors", showlegend=False,
    ))
    fig.update_layout(
        title="APPENDIX — RadViz (exploratory)",
        height=520, margin=dict(t=80), template="plotly_white",
        xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
    )
    return fig


appendix_figs = [
    ("A1_parallel_coordinates.png", fig_parallel_coords(df)),
    ("A2_treemap.png", fig_treemap(df)),
    ("A3_pixel_matrix.png", fig_pixel_matrix(df)),
    ("A4_splom.png", fig_splom(df)),
    ("A5_parallel_sets.png", fig_parallel_sets(df)),
    ("A6_radviz.png", fig_radviz(df)),
]

for name, fig in appendix_figs:
    fig.show()
    save_static(fig, APPENDIX_DIR / name, width=1000, height=560)

print("Appendix figures written under", APPENDIX_DIR)


/var/folders/ql/97jfk1x152v1dwrlkfh3vft00000gn/T/ipykernel_15904/2518965078.py:22: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True).codes
/var/folders/ql/97jfk1x152v1dwrlkfh3vft00000gn/T/ipykernel_15904/2518965078.py:87: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  d["instr_ord"] = pd.Categorical(d["instrucao"], categories=INSTR_ORDER, ordered=True).codes
/var/folders/ql/97jfk1x152v1dwrlkfh3vft00000gn/T/ipykernel_15904/2518965078.py:154: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  d["instr_ord"] = pd.Categorical(

saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A1_parallel_coordinates.png


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A2_treemap.png


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A3_pixel_matrix.png


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A4_splom.png


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A5_parallel_sets.png


saved /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures/A6_radviz.png
Appendix figures written under /Users/etorebraga/Code/masters-curriculum/courses/ft105a-information-visualization/project/trabalho-1-pnad-multidim/code/output/figures


### Appendix note

- Official graded techniques remain: bubble scatter, heatmap small-multiples, Sankey.  
- Primary analysis code and Portuguese findings: `pnad_analise_ponderada.ipynb` (**Raphael Pizzi**).  
- Exploratory authoring history: `pnad_dataviz.ipynb`.  
- Static PNGs for the PDF: `entregas/figures/` (main trio) and `code/output/figures/` (appendix).
